# Equations and root finding

```{admonition} Learning outcomes
After working through this chapter, you should be able to:

1. formulate an equation as a root-finding problem $f(x)=0$
2. explain and implement the bisection method and Newton's method
3. use tolerances and maximum iteration counts to control a numerical calculation
4. assess strengths, limitations and convergence of different root-finding methods
5. use root solvers from SciPy for chemical problems
```

Many chemical problems end with an equation that must be solved. Some can be solved analytically, but numerical methods are often more practical when the expressions become complicated. An equation $g(x)=h(x)$ can be rewritten as $f(x)=g(x)-h(x)=0$. A **root** is an $x$ value for which the function value is zero.


## A chemical example: pH of a weak acid

Consider a 0.010 M solution of acetic acid. For a monoprotic weak acid with analytical concentration $C$,

$$[\mathrm{A^-}]=C\frac{K_a}{[\mathrm{H_3O^+}]+K_a}.$$

Charge balance and the ionic product of water give

$$[\mathrm{H_3O^+}]=[\mathrm{A^-}]+[\mathrm{OH^-}],\qquad [\mathrm{OH^-}]=\frac{K_w}{[\mathrm{H_3O^+}]}.$$

If $h=[\mathrm{H_3O^+}]$, the problem becomes

$$f(h)=h-C\frac{K_a}{h+K_a}-\frac{K_w}{h}.$$

The pH is obtained when $f(h)=0$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C = 0.010
Ka = 1.8e-5
Kw = 1.0e-14

def charge_balance(h):
    return h - C*Ka/(h + Ka) - Kw/h

h = np.logspace(-6, -1, 500)
plt.semilogx(h, charge_balance(h))
plt.axhline(0)
plt.xlabel("[H3O+] (mol/L)")
plt.ylabel("f(h)")
plt.show()


## The bisection method

If a continuous function changes sign between $a$ and $b$, there is at least one root in the interval. The bisection method repeatedly halves this interval. It is not especially fast, but it is robust when a valid bracket is known.


In [ ]:
def bisection(f, a, b, tolerance=1e-10, max_iterations=100):
    fa = f(a)
    fb = f(b)
    if fa * fb > 0:
        raise ValueError("The interval must bracket a root.")
    for _ in range(max_iterations):
        midpoint = (a + b) / 2
        fm = f(midpoint)
        if abs(fm) < tolerance or (b - a)/2 < tolerance:
            return midpoint
        if fa * fm < 0:
            b = midpoint
        else:
            a = midpoint
            fa = fm
    raise RuntimeError("Maximum number of iterations reached.")

h_root = bisection(charge_balance, 1e-6, 1e-1)
print("pH =", -np.log10(h_root))


## Newton's method

Newton's method uses the local tangent to estimate the next root:

$$x_{n+1}=x_n-\frac{f(x_n)}{f'(x_n)}.$$

It can converge very rapidly near a simple root, but it requires a derivative and a sensible starting value. A poor starting value can lead to slow convergence, divergence or convergence to a different root.


In [ ]:
def newton(f, derivative, x0, tolerance=1e-10, max_iterations=50):
    x = x0
    for _ in range(max_iterations):
        slope = derivative(x)
        if slope == 0:
            raise ZeroDivisionError("Derivative became zero.")
        x_new = x - f(x)/slope
        if abs(x_new - x) < tolerance:
            return x_new
        x = x_new
    raise RuntimeError("Maximum number of iterations reached.")

def charge_balance_derivative(h):
    return 1 + C*Ka/(h + Ka)**2 + Kw/h**2

h_root_newton = newton(charge_balance, charge_balance_derivative, 1e-3)
print("pH =", -np.log10(h_root_newton))


## SciPy root solvers

In scientific work we normally use tested library implementations rather than writing our own solver every time. Implementing the methods ourselves is still useful because it makes tolerances, convergence and failure modes visible.


In [ ]:
from scipy.optimize import brentq, root_scalar

h_brent = brentq(charge_balance, 1e-6, 1e-1)
solution = root_scalar(charge_balance, bracket=[1e-6, 1e-1], method="brentq")
print("pH =", -np.log10(h_brent))
print("Converged:", solution.converged)


## Choosing a method

| Method | Main advantage | Main limitation |
|---|---|---|
| Bisection | robust when a root is bracketed | relatively slow |
| Newton | very fast near a root | needs derivative and good initial guess |
| Brent / `root_scalar` | robust and efficient | still needs a suitable formulation or bracket |

```{admonition} Good numerical practice
:class: important
A numerical answer should be checked chemically as well as mathematically. Verify units, physically meaningful ranges, residual $|f(x)|$, and sensitivity to tolerances or starting values.
```

## Exercises

1. Solve $x^3-2x-5=0$ with bisection and with a SciPy solver.
2. Repeat the weak-acid calculation at another concentration and compare with $[\mathrm{H_3O^+}]\approx\sqrt{K_aC}$.
3. Construct an example where Newton's method performs poorly and explain why.
